# Causal Tracing (Alt)

Graph-first notebook for `src/causal_trace/alt_trace.py`.

This is intentionally different from `causal_tracing.ipynb`: it restores only the last subject token, averages over repeated noise samples, then ranks layers with the middle-third fallback when the signal is noisy. Use the same `MODEL_CONFIG` and prompt counts in both notebooks when comparing them.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'src' / 'main.py').exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')


In [ ]:
import logging

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from hydra import compose, initialize_config_dir

from src.handlers.rome import ModelHandler
from src.common.loading import load_dataset
from src.causal_trace.causal_trace import filter_dataset, preprocess_prompt
from src.causal_trace.alt_trace import (
    trace_prompt,
    select_layers,
    _ensure_noise_multiplier,
    _save_results,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.grid'] = True
print('Imports OK')


## 2. Load Config, Model, And Dataset

Set `MODEL_CONFIG` to a file from `src/config/model/` without `.yaml`. `NUM_TRACE_RUNS` controls repeated noise samples per prompt.


In [ ]:
MODEL_CONFIG = 'gpt2-large'
NUM_PROMPTS = 3
NUM_TRACE_RUNS = 10

config_dir = str(ROOT / 'src' / 'config')
with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(
        config_name='latium',
        overrides=[
            'command=alt_trace',
            f'model={MODEL_CONFIG}',
            f'generation.num_of_runs={NUM_PROMPTS}',
            f'generation.num_trace_runs={NUM_TRACE_RUNS}',
        ],
    )

print(f'Model: {cfg.model.name}')
print(f'Command: {cfg.command.name}')
print(f'Noise multiplier: {cfg.model.get("corruption_noise_multiplier", "auto")}')


In [ ]:
handler = ModelHandler(cfg)
print(f'Loaded {cfg.model.name} with {handler.num_of_layers} layers')

dataset = load_dataset(cfg)
df_dataset = filter_dataset(dataset['requested_rewrite'])
print(f'Dataset rows: {len(df_dataset)}')

_ensure_noise_multiplier(handler, cfg, df_dataset)
print(f'Noise multiplier in use: {handler._noise_multiplier:.6f}')


## 3. Run Alt Traces

In [ ]:
results = []
total = 0
failed = 0

for prompt_dict in df_dataset.itertuples():
    if len(results) >= NUM_PROMPTS:
        break
    total += 1

    preprocessed = preprocess_prompt(handler, prompt_dict)
    if preprocessed is None:
        failed += 1
        continue

    prompt_ids, subject_positions = preprocessed
    result = trace_prompt(
        handler,
        prompt_ids,
        subject_positions,
        prompt_dict.target_true['str'],
        num_runs=NUM_TRACE_RUNS,
    )

    if result is None:
        failed += 1
        print(f'SKIP clean-token mismatch: {prompt_dict.subject!r} -> {prompt_dict.target_true["str"]!r}')
        continue

    result.prompt_idx = prompt_dict.Index
    result.subject = prompt_dict.subject
    results.append(result)
    peak = int(np.argmax(result.per_layer_probs))
    print(
        f'OK {len(results):02d}: {result.subject!r} -> {result.target!r}  '
        f'clean={result.clean_prob:.4f} corrupt={result.corrupt_prob:.4f} peak=L{peak}'
    )

print(f'Done: {len(results)} successful, {failed} failed, {total} attempted')


## 4. Prompt Gallery

Each curve is the mean restoration probability across repeated noise samples for one prompt. The dashed line marks the prompt's peak layer.


In [ ]:
if not results:
    raise RuntimeError('No successful traces. Try increasing NUM_PROMPTS or changing MODEL_CONFIG.')

cols = min(len(results), 3)
rows = int(np.ceil(len(results) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(5.2 * cols, 3.8 * rows), squeeze=False)
layers = np.arange(handler.num_of_layers)

for ax, result in zip(axes.ravel(), results):
    probs = result.per_layer_probs
    peak = int(np.argmax(probs))
    ax.plot(layers, probs, marker='o', color='steelblue')
    ax.axhline(result.clean_prob, color='seagreen', linestyle=':', label='clean prob')
    ax.axhline(result.corrupt_prob, color='gray', linestyle=':', label='corrupt prob')
    ax.axvline(peak, color='crimson', linestyle='--', label=f'peak L{peak}')
    ax.set_title(f'{result.subject} -> {result.target}', fontsize=9)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Target-token probability')
    ax.set_xlim(-0.5, handler.num_of_layers - 0.5)
    ax.legend(fontsize=8)

for ax in axes.ravel()[len(results):]:
    ax.axis('off')

fig.suptitle(f'Alt causal tracing: {cfg.model.name} ({NUM_TRACE_RUNS} noise runs/prompt)', fontsize=13)
plt.tight_layout()
plt.show()


## 5. Run-Level Graphs And Layer Selection

In [ ]:
curves = np.stack([result.per_layer_probs for result in results], axis=0)
avg_probs = curves.mean(axis=0)
std_probs = curves.std(axis=0)
selection = select_layers(avg_probs, handler.num_of_layers)
peak = int(np.argmax(avg_probs))
mid_start = handler.num_of_layers // 3
mid_end = 2 * handler.num_of_layers // 3

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

heat = axes[0].imshow(curves, aspect='auto', cmap='magma')
axes[0].set_title('Prompt x layer restoration heatmap')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Prompt')
axes[0].set_yticks(range(len(results)))
axes[0].set_yticklabels([f'{r.subject} -> {r.target}' for r in results], fontsize=8)
for idx, curve in enumerate(curves):
    axes[0].plot(int(np.argmax(curve)), idx, marker='x', color='white', markersize=7)
fig.colorbar(heat, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].axvspan(mid_start, mid_end, alpha=0.12, color='orange', label='middle third')
axes[1].plot(layers, avg_probs, marker='o', color='steelblue', label='mean')
axes[1].fill_between(layers, avg_probs - std_probs, avg_probs + std_probs, color='steelblue', alpha=0.18, label='1 std')
axes[1].axvline(peak, color='crimson', linestyle='--', label=f'peak L{peak}')
axes[1].axvline(selection.best_layer, color='darkgreen', linestyle='-.', label=f'selected L{selection.best_layer}')
axes[1].set_title(
    f'Average restoration curve | quality={selection.signal_quality}, fallback={selection.used_middle_third_fallback}'
)
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Target-token probability')
axes[1].set_xlim(-0.5, handler.num_of_layers - 0.5)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
print(selection.summary())


## 6. Candidate Table And Optional Save

In [ ]:
summary = pd.DataFrame(
    {
        'prompt_idx': result.prompt_idx,
        'subject': result.subject,
        'target': result.target,
        'clean_prob': result.clean_prob,
        'corrupt_prob': result.corrupt_prob,
        'peak_layer': int(np.argmax(result.per_layer_probs)),
        'peak_prob': float(np.max(result.per_layer_probs)),
    }
    for result in results
)

candidates = pd.DataFrame(
    {
        'rank': candidate.rank,
        'layer': candidate.layer,
        'restoration_prob': candidate.restoration_prob,
        'in_middle_third': candidate.in_middle_third,
    }
    for candidate in selection.candidates[:10]
)

summary, candidates


In [ ]:
SAVE = False

if SAVE:
    csv_path, json_path = _save_results(results, avg_probs, selection, cfg)
    print(f'CSV:  {csv_path}')
    print(f'JSON: {json_path}')
else:
    print('Set SAVE = True to write CSV + JSON to analysis_out/')
